In [1]:
import mediapipe as mp 
import cv2
import pyautogui
import numpy as np
import uuid
import math

In [2]:
cap = cv2.VideoCapture(0)
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

screen_w, screen_h = pyautogui.size()

cam_w = int(cap.get(3))
cam_h = int(cap.get(4))

click_clicked = False
click_clicked = False 
double_click_clicked = False 

smoothening = 5 
plocX, plocY = 0, 0  
clocX, clocY = 0, 0 

smoothening = 7
frameR = 100

In [3]:
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.7) as hands:
    while cap.isOpened():
        ret , frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = cv2.flip(image, 1)
        
        cv2.rectangle(image, (frameR, frameR), (cam_w - frameR, cam_h - frameR), (255, 0, 255), 2)
        
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.multi_hand_landmarks : 
            for num, hand in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(image , hand, mp_hands.HAND_CONNECTIONS,
                                         mp_drawing.DrawingSpec(color = (25,175,221), thickness=2, circle_radius = 3),
                                         mp_drawing.DrawingSpec(color = (25,175,221), thickness=2, circle_radius = 2),
                                         )
                
                h, w, c = image.shape
                
                # استخراج إحداثيات رؤوس الأصابع ومفاصلها السفلية (لمعرفة هل الإصبع مرفوع أم لا)
                x_thumb, y_thumb = int(hand.landmark[4].x * w), int(hand.landmark[4].y * h)
                
                x_index, y_index = int(hand.landmark[8].x * w), int(hand.landmark[8].y * h)
                y_index_pip = int(hand.landmark[6].y * h) # المفصل السفلي للسبابة
                
                x_middle, y_middle = int(hand.landmark[12].x * w), int(hand.landmark[12].y * h)
                y_middle_pip = int(hand.landmark[10].y * h) # المفصل السفلي للوسطى
                
                # ==========================================
                # التحقق من حالة الأصابع (مرفوعة أم مطوية)
                # (في OpenCV، النقطة الأعلى في الشاشة لها قيمة Y أصغر)
                # ==========================================
                index_is_up = y_index < y_index_pip
                middle_is_up = y_middle < y_middle_pip

                # حساب الإحداثيات والتنعيم دائماً لتتبع الحركة
                screen_x = np.interp(x_index, (frameR, cam_w - frameR), (0, screen_w))
                screen_y = np.interp(y_index, (frameR, cam_h - frameR), (0, screen_h))
                
                clocX = plocX + (screen_x - plocX) / smoothening
                clocY = plocY + (screen_y - plocY) / smoothening
                
                # ==========================================
                # الوضع الأول: وضع الحركة والنقر (السبابة مرفوعة، الوسطى مطوية ☝️)
                # ==========================================
                if index_is_up and not middle_is_up:
                    # 1. تحريك الماوس
                    pyautogui.moveTo(clocX, clocY)
                    
                    # 2. النقر العادي (الإبهام + السبابة 🤏)
                    distance_single = math.hypot(x_index - x_thumb, y_index - y_thumb)
                    if distance_single < 25: # تم تقليل المسافة لتفادي النقر الخطأ
                        cv2.circle(image, (x_index, y_index), 15, (0, 255, 0), cv2.FILLED) 
                        if not click_clicked:
                            pyautogui.click()
                            click_clicked = True
                    else:
                        click_clicked = False
                        
                    # 3. النقر المزدوج (الإبهام + الوسطى 🤌) -> تم تغيير الإيماءة لتفادي التداخل!
                    distance_double = math.hypot(x_middle - x_thumb, y_middle - y_thumb)
                    if distance_double < 25:
                        cv2.circle(image, (x_middle, y_middle), 15, (0, 0, 255), cv2.FILLED) 
                        if not double_click_clicked:
                            pyautogui.doubleClick()
                            double_click_clicked = True
                    else:
                        double_click_clicked = False

                # ==========================================
                # الوضع الثاني: وضع التمرير / Scroll (السبابة والوسطى مرفوعتان ✌️)
                # ==========================================
                elif index_is_up and middle_is_up:
                    # الماوس لا يتحرك هنا، بل نحسب فرق الارتفاع للتمرير
                    scroll_amount = (plocY - clocY)
                    
                    # نتجاهل الحركات الصغيرة جداً لتفادي الاهتزاز (Deadzone)
                    if abs(scroll_amount) > 2:
                        # نضرب القيمة في 3 لزيادة سرعة التمرير (يمكنك تغيير الرقم)
                        pyautogui.scroll(int(scroll_amount * 3))
                        cv2.putText(image, "Scrolling...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

                # تحديث المواقع للإطار القادم (مهم جداً ليظل التمرير والتنعيم دقيقين)
                plocX, plocY = clocX, clocY

        cv2.imshow("Hand Tracking", image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [4]:
#os.mkdir("Output Image")

In [5]:
#mp_hands.HAND_CONNECTIONS